In [ ]:
!pip install google-generativeai -q

In [ ]:
# ============================================================
#   SWIGGY CUSTOMER CARE AI AGENT
#   Built with: Gemini Free API + Prompt Engineering
#   Run this in Google Colab — cell by cell
# ============================================================


# ── CELL 1: Install ──────────────────────────────────────────
# !pip install google-generativeai -q


# ── CELL 2: Full Code ────────────────────────────────────────

import google.generativeai as genai

# ── 1. YOUR API KEY ──────────────────────────────────────────
# Get free key from: https://aistudio.google.com/app/apikey
API_KEY = "YOUR_API_KEY"
genai.configure(api_key=API_KEY)


# ── 2. THE SYSTEM PROMPT (this is the prompt engineering) ────
SYSTEM_PROMPT = """
You are "Swiggy Sathi" — Swiggy India's official AI customer care assistant.

YOUR JOB:
Help customers ONLY with problems related to:
- Food orders (status, delays, missing orders)
- Wrong or missing items in delivery
- Refunds and cancellations
- Delivery issues (late, wrong address, delivery partner)
- Restaurant queries (menu, availability, timings)
- Payment problems (failed payment, double charge, Swiggy Money)
- Offers, coupons, Swiggy One subscription
- App issues (login, address, payment method)
- Food quality complaints
- Allergy or dietary requirement queries (vegan, Jain, gluten-free)

STRICT RULE — OFF-TOPIC QUERIES:
If anyone asks about ANYTHING other than food, delivery, or Swiggy —
DO NOT answer the question. Instead reply EXACTLY:

"Haha, I wish I could help with that! 😄 But I'm Swiggy Sathi —
I only know food, orders, and deliveries. For [their topic], you'll
need a different assistant! 🍕
Can I help you with anything on Swiggy instead?"

NEVER help with: general knowledge, cricket, movies, coding homework,
medical advice, other apps, politics, weather, or anything non-food.

SWIGGY KNOWLEDGE:
- Refund to bank/card: 5–7 business days
- Refund to Swiggy Money: within 24 hours (recommend this)
- Cancellation before restaurant accepts = 100% refund
- Cancellation after restaurant accepts = case by case
- Average delivery time: 30–45 minutes
- To escalate: App → Help → My Orders → Select Order → Report Issue
- Swiggy One members get free delivery + priority support

TONE:
- Friendly, warm, helpful — like a friend who works at Swiggy
- Natural Indian English ("No worries!", "Got it!", "Sure thing!")
- For complaints: empathize first, then solve
- Keep responses short — 3 to 5 sentences max
- End with: "Anything else I can help you with? 😊"
- Use emojis lightly: 🍕 🛵 💰 ✅
"""

# ── 3. NON-FOOD KEYWORDS (instant rejection, no API call) ────
OFF_TOPIC_WORDS = [
    "cricket", "ipl", "football", "movie", "film", "song",
    "python", "javascript", "code", "homework", "essay", "project",
    "weather", "news", "politics", "election", "government",
    "relationship", "love", "marriage", "job", "career", "resume",
    "doctor", "symptoms", "medicine", "hospital", "health",
    "amazon", "flipkart", "zomato", "myntra", "meesho",
    "bitcoin", "crypto", "stock", "investment", "share market",
    "history", "geography", "science", "math", "physics",
    "chat gpt", "chatgpt", "openai", "who made you", "who created you"
]

def is_off_topic(query):
    query_lower = query.lower()
    for word in OFF_TOPIC_WORDS:
        if word in query_lower:
            return True
    return False


# ── 4. THE AGENT CLASS ────────────────────────────────────────
class SwiggyCareAgent:

    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.5-flash")
        self.chat_history = []   # stores full conversation
        self.query_count = 0

    def chat(self, user_message):
        self.query_count += 1

        # Step 1: Fast check — obviously off-topic?
        if is_off_topic(user_message):
            return self._off_topic_response()

        # Step 2: Build messages with system prompt + history
        messages = [
            {"role": "user",      "parts": [SYSTEM_PROMPT]},
            {"role": "model",     "parts": ["Understood! I'm Swiggy Sathi, ready to help with food and delivery queries only. 🍕"]},
        ]

        # Add conversation history (last 6 exchanges = 12 messages)
        for msg in self.chat_history[-12:]:
            messages.append(msg)

        # Add current user message
        messages.append({"role": "user", "parts": [user_message]})

        # Step 3: Call Gemini
        try:
            response = self.model.generate_content(messages)
            reply = response.text.strip()

            # Save to history
            self.chat_history.append({"role": "user",  "parts": [user_message]})
            self.chat_history.append({"role": "model", "parts": [reply]})

            return reply

        except Exception as e:
            return f"Oops! Something went wrong on our end. Please try again in a moment. 🙏\n(Error: {str(e)})"

    def _off_topic_response(self):
        return (
            "Haha, I wish I could help with that! 😄\n"
            "But I'm Swiggy Sathi — I only know food, orders, and deliveries.\n"
            "Can I help you with anything on Swiggy instead? 🍕"
        )

    def reset(self):
        self.chat_history = []
        print("\n🔄 Conversation reset. Starting fresh!\n")


# ── 5. PRETTY PRINT HELPERS ───────────────────────────────────
def print_banner():
    print("=" * 55)
    print("  🍕  SWIGGY SATHI — AI Customer Care Agent")
    print("  Powered by Gemini Free API + Prompt Engineering")
    print("=" * 55)
    print("  Type your question below.")
    print("  Type 'reset' to clear history | 'quit' to exit")
    print("-" * 55)

def print_bot(text):
    print(f"\n🤖 Swiggy Sathi:\n{text}\n")
    print("-" * 55)

def print_user(text):
    print(f"\n👤 You: {text}")


# ── 6. MAIN CHAT LOOP ─────────────────────────────────────────
def run():
    print_banner()
    agent = SwiggyCareAgent()

    # Opening message
    print_bot(
        "Namaste! 🙏 I'm Swiggy Sathi, your Swiggy customer care assistant!\n"
        "I can help you with orders, refunds, delivery issues, restaurants,\n"
        "payments, and anything else related to your Swiggy experience.\n"
        "How can I help you today? 😊"
    )

    while True:
        try:
            user_input = input("👤 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\n👋 Thanks for using Swiggy Sathi. Have a great meal! 🍕")
            break

        if not user_input:
            continue

        if user_input.lower() == "quit":
            print("\n👋 Thanks for using Swiggy Sathi. Have a great meal! 🍕")
            break

        if user_input.lower() == "reset":
            agent.reset()
            continue

        print_user(user_input)
        reply = agent.chat(user_input)
        print_bot(reply)


# ── 7. RUN IT ─────────────────────────────────────────────────
if __name__ == "__main__":
    run()

  🍕  SWIGGY SATHI — AI Customer Care Agent
  Powered by Gemini Free API + Prompt Engineering
  Type your question below.
  Type 'reset' to clear history | 'quit' to exit
-------------------------------------------------------

🤖 Swiggy Sathi:
Namaste! 🙏 I'm Swiggy Sathi, your Swiggy customer care assistant!
I can help you with orders, refunds, delivery issues, restaurants,
payments, and anything else related to your Swiggy experience.
How can I help you today? 😊

-------------------------------------------------------
👤 You: Prachi Pandey

👤 You: Prachi Pandey

🤖 Swiggy Sathi:
Hey Prachi! Welcome to Swiggy Sathi. How can I help you with your Swiggy order today? 😊

-------------------------------------------------------
👤 You: Prachi Pandey

👤 You: Prachi Pandey

🤖 Swiggy Sathi:
Hi Prachi! That's your name, got it. How can I assist you with your Swiggy experience today? Do you have a question about an order, a refund, or anything else Swiggy-related? 😊

---------------------------------

In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 69.3 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import google.generativeai as genai

# ---------------- API KEY ----------------
API_KEY = "YOUR_API_KEY"

genai.configure(api_key=API_KEY)

# ---------------- SYSTEM PROMPT ----------------
SYSTEM_PROMPT = """
You are Swiggy Sathi — Swiggy India's AI support assistant.

Help ONLY with:
- food orders
- refunds
- delivery issues
- restaurant queries
- payment problems

If user asks unrelated things, politely refuse.
Keep replies friendly and short.
"""

# ---------------- OFF TOPIC FILTER ----------------
OFF_TOPIC = [
    "cricket", "movie", "python", "code",
    "weather", "politics", "bitcoin"
]

def is_off_topic(text):
    text = text.lower()
    return any(word in text for word in OFF_TOPIC)

# ---------------- AI AGENT ----------------
model = genai.GenerativeModel("gemini-2.5-flash")

st.set_page_config(
    page_title="Swiggy Sathi",
    page_icon="🍕"
)

st.title("🍕 Swiggy Sathi AI")
st.write("Your AI-powered Swiggy Customer Care Assistant")

if "messages" not in st.session_state:
    st.session_state.messages = []

# Show old chats
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# User input
prompt = st.chat_input("Ask your Swiggy question...")

if prompt:

    # Show user message
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message("user"):
        st.markdown(prompt)

    # Off topic check
    if is_off_topic(prompt):
        reply = (
            "Haha 😄 I'm Swiggy Sathi.\n"
            "I only help with food orders and deliveries 🍕"
        )

    else:
        try:
            response = model.generate_content(
                SYSTEM_PROMPT + "\nUser: " + prompt
            )
            reply = response.text

        except Exception as e:
            reply = f"Error: {str(e)}"

    # Show assistant reply
    with st.chat_message("assistant"):
        st.markdown(reply)

    st.session_state.messages.append({
        "role": "assistant",
        "content": reply
    })

Overwriting app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-05-28 16:27:32.530 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.150.131.52:8501

  Stopping...
^C


In [ ]:
from pyngrok import ngrok
import threading
import os

# Start Streamlit
def run():
    os.system("streamlit run app.py --server.port 8501")

thread = threading.Thread(target=run)
thread.start()

# Create public URL
public_url = ngrok.connect(8501)

print("Your App URL:")
print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-05-28T16:29:27+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-28T16:29:27+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-28T16:29:27+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
!ngrok config add-authtoken 3EMQPCBwsXqoAq97GQmu1NPinoy_7SUphvfZ1r2QiQHZ6qQdk

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
import threading
import os

def run():
    os.system("streamlit run app.py --server.port 8501")

thread = threading.Thread(target=run)
thread.start()

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://situation-draw-corned.ngrok-free.dev" -> "http://localhost:8501"
